In [1]:
import pandas as pd
import numpy as np
from cliffs_delta import cliffs_delta
from scipy.stats import mannwhitneyu

from scripts.dependency_extractor import DependencyExtractor
from datetime import datetime, timezone

### Projects that use Dependabot

### Apply secletion criteria on project to study (Do not exected, as we did it at the begining of the project)

In [16]:
df_prs = pd.read_csv("./data/11k_repository_pull_requests_full.csv")
# df.drop_duplicates(subset=['repo'], inplace=True)
df_prs["repo_created_at"] = pd.to_datetime(df_prs["repo_created_at"]).dt.tz_localize(None)
df_prs["repo_last_committed_date"] = pd.to_datetime(df_prs["repo_last_committed_date"]).dt.tz_localize(None)

date_string = "2025-07-07 22:32:59"
date_format = "%Y-%m-%d %H:%M:%S"
df_prs["repo_last_update"] = (datetime.strptime(date_string, date_format).replace(tzinfo=None) - df_prs['repo_last_committed_date']).dt.total_seconds() / (3600 * 24)

df_prs['repo_age'] = (df_prs['repo_last_committed_date'] - df_prs['repo_created_at']).dt.days
# df_prs.drop(columns=['body'], inplace=True, errors='ignore')

keywords = [
    'tutorial', 'docs', 'documentation', 'lab', 'presentation',
    'exercise', 'bootcamp', 'hackathon', 'assignment', 'practice',
    'resume', 'template', 'talks', 'udemy', 'sample', 'notebook',
    'cookbook', 'guide', 'note', 'course', 'example', 'educational',
    'survey', 'dataset', 'academic', 'academia', 'academy', 'interview',
    'lesson', 'my-', 'my_', 'notebook', 'portfolio', 'leetcode', 'neetcode',
    'challenge'
]

# repos_to_keep = [
#     'indentlabs/notebook', 'urbandroid-team/dont-kill-my-app', 'fastify/light-my-request', 'pin3da/notebook-generator', 'dataflownb/df_prsnotebook', 'manovotny/eslint-config-get-off-my-lawn', 'my-telegram-bots/Pixiv_bot', 'morganhvidt/find-my-blocks', 'anubhavsrivastava/blame-my-network', 'GoogleChromeLabs/how-fugu-is-my-browser', 'ArelySkywalker/Six-Gummy-Bears-and-some-Scotch', 'scimmyjs/scimmy-routers', 'Sammy-T/gotepad', 'Cweili/koa-router-find-my-way', 'NamVr/Chat-Economy-Bot', 'XenoX/my-oc'
# ]
# excluded_repos = ['choyiny/cscc09.com']
#
# # Convert keywords to lowercase for case-insensitive matching
keywords_lower = [kw.lower() for kw in keywords]

# Select project that have more than 10 dependabot PRs
dep_pr_counts = df_prs.groupby('repo').size().reset_index(name='dep_pr_count')
repo_more_10_dep_prs = dep_pr_counts.loc[dep_pr_counts['dep_pr_count'] >= 10, 'repo'].tolist()

df_prs = df_prs[
    (df_prs['stargazerCount'] >= 10) &
    (df_prs['repoAge'] >= 180) &
    (df_prs['primaryLanguage'] == "JavaScript") &
    (df_prs['isArchived'] == False) &
    (df_prs['isTemplate'] == False) &
    (df_prs['isFork'] == False) &
    (df_prs['commitHistoryCount'] >= 20) &
    (
        ((~df_prs['repo'].str.lower().str.contains('|'.join(keywords_lower), na=False))
        & (~df_prs['description'].str.lower().str.contains('|'.join(keywords_lower), na=False)))|
        df_prs['repo'].isin(repo_more_10_dep_prs)
    )
]

### Select a representative sample

In [ ]:
# Step 1: Compute medians for the whole dataset (across repos)
columns = ['repo', 'stargazer_count', 'repo_commit_count', 'mentionable_users_count']
df_repos = df_prs[columns].drop_duplicates(subset=['repo']).copy()
star_median = df_repos['stargazer_count'].median()
commit_median = df_repos['repo_commit_count'].median()
contributor_median = df_repos['mentionable_users_count'].median()

# Step 2: Create star_label, commit_label, contributor_label
df_prs['star_label'] = df_prs['stargazer_count'].apply(lambda x: 'less-mature' if x < star_median else 'mature')
df_prs['commit_label'] = df_prs['repo_commit_count'].apply(lambda x: 'less-mature' if x < commit_median else 'mature')
df_prs['contributor_label'] = df_prs['mentionable_users_count'].apply(lambda x: 'less-mature' if x < contributor_median else 'mature')

# Step 3: Merge back into original df
df_prs = df_prs.merge(dep_pr_counts, on='repo', how='left')

# Step 4: Compute median for dependabot PRs
dep_pr_median = dep_pr_counts['dep_pr_count'].median()

# Step 6: Assign dep_pr_label
df_prs['dep_pr_label'] = df_prs['dep_pr_count'].apply(lambda x: 'less-mature' if x < dep_pr_median else 'mature')

repo_df = df_prs.groupby('repo').agg({
    'stargazer_count': 'first',
    'repo_commit_count': 'first',
    'mentionable_users_count': 'first'
}).reset_index()

dep_pr_counts = df_prs.groupby('repo').size().reset_index(name='dependabot_pr_count')
repo_df = repo_df.merge(dep_pr_counts, on='repo', how='left')
repo_df['dependabot_pr_count'] = repo_df['dependabot_pr_count'].fillna(0)

repo_df['star_label'] = repo_df['stargazer_count'].apply(
    lambda x: 'less' if x < repo_df['stargazer_count'].median() else 'more')
repo_df['commit_label'] = repo_df['repo_commit_count'].apply(
    lambda x: 'less' if x < repo_df['repo_commit_count'].median() else 'more')
repo_df['contributor_label'] = repo_df['mentionable_users_count'].apply(
    lambda x: 'less' if x < repo_df['mentionable_users_count'].median() else 'more')
repo_df['dep_pr_label'] = repo_df['dependabot_pr_count'].apply(
    lambda x: 'less' if x < repo_df['dependabot_pr_count'].median() else 'more')

# ---- Step 5: Sample X repos maintaining original proportions ----
def stratified_repo_sample_proportional(df, label_col, total_sample_size, random_state=42):
    # Calculate proportions of each category
    proportions = df[label_col].value_counts(normalize=True)

    # Calculate how many to sample from each group
    samples_per_group = (proportions * total_sample_size).round().astype(int)

    # Adjust total to exactly match the requested sample size
    # (due to rounding, we might be off by 1-2)
    diff = total_sample_size - samples_per_group.sum()
    if diff != 0:
        # Add the difference to the largest group
        largest_group = proportions.idxmax()
        samples_per_group[largest_group] += diff

    # Sample from each group
    samples = []
    for group, count in samples_per_group.items():
        if count > 0:
            group_sample = df[df[label_col] == group].sample(n=count, random_state=random_state)
            samples.append(group_sample)

    return pd.concat(samples)

# Set your desired total sample size
X = 348

# Sampled repo subsets with total size X, maintaining original proportions
sampled_star_repos = stratified_repo_sample_proportional(repo_df, 'star_label', X)
sampled_commit_repos = stratified_repo_sample_proportional(repo_df, 'commit_label', X)
sampled_contrib_repos = stratified_repo_sample_proportional(repo_df, 'contributor_label', X)
sampled_dep_pr_repos = stratified_repo_sample_proportional(repo_df, 'dep_pr_label', X)

# ---- Step 6 remains the same ----
prs_star = df_prs[df_prs['repo'].isin(sampled_star_repos['repo'])]
prs_commit = df_prs[df_prs['repo'].isin(sampled_commit_repos['repo'])]
prs_contrib = df_prs[df_prs['repo'].isin(sampled_contrib_repos['repo'])]
prs_dep = df_prs[df_prs['repo'].isin(sampled_dep_pr_repos['repo'])]

# ---- Step 7 save the data to CSV files
prs_star2 = pd.read_csv("./data/prs_star.csv")
prs_star = pd.merge(prs_star, prs_star2[['repo', 'id', 'pr_category']], on=['repo', 'id'], how='left')

prs_commit2 = pd.read_csv("./data/prs_commit.csv")
prs_commit = pd.merge(prs_commit, prs_commit2[['repo', 'id', 'pr_category']], on=['repo', 'id'], how='left')

prs_contrib2 = pd.read_csv("./data/prs_contrib.csv")
prs_contrib = pd.merge(prs_contrib, prs_contrib2[['repo', 'id', 'pr_category']], on=['repo', 'id'], how='left')

prs_dep2 = pd.read_csv("./data/prs_dep.csv")
prs_dep = pd.merge(prs_dep, prs_dep2[['repo', 'id', 'pr_category']], on=['repo', 'id'], how='left')